Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드 스크립트다.

용도: 제미나이(Gemini ) API 프롬프트 전송 전, Cloud DLP API(Sensitive Data Protection )를 연동해 민감한 개인정보(PII ) 유출을 실시간 감지하고 마스킹(Redaction ) 처리하는 시나리오를 가이드 및 테스트한다.

## 제미나이 API 개인정보 보호 안심 필터 (Gemini API Sensitive Data Filter )

### 1. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색하고 감지 대상 개인정보 유형과 예제 프롬프트를 변수로 선언한다.

In [ ]:
import google.auth

DLP_INFO_TYPES = "PHONE_NUMBER,EMAIL_ADDRESS,KOREA_RRN"
DLP_MASK_CHARACTER = "*"
TEST_PROMPT = "안녕하세요. 제 연락처는 010-1234-5678이고 주민등록번호는 900101-1234567입니다. gcp-user@example.com 으로 제미나이 3.5 분석 자료를 보내주세요."

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id"


### 2. Cloud DLP API 자가 활성화 및 마스킹 검증 가이드

Cloud DLP API가 비활성화되어 있는 경우 자가 활성화를 시도하고, curl 또는 Python requests 라이브러리를 통해 Cloud DLP API의 `content:deidentify` 엔드포인트를 호출하여 프롬프트 내부 개인정보를 기호로 마스킹 변환한다.

(구글 클라우드 공식 참조 주소: https://cloud.google.com/sensitive-data-protection/docs/redacting-sensitive-data )

```bash
# 1) Cloud DLP API 활성화
gcloud services enable dlp.googleapis.com --project="YOUR_PROJECT_ID"

# 2) Cloud DLP API deidentify 엔드포인트를 호출해 프롬프트 마스킹 변환
curl -s -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -d '{"item": {"value": "안녕하세요. 제 연락처는 010-1234-5678입니다."}, "deidentifyConfig": {"infoTypeTransformations": {"transformations": [{"infoTypes": [{"name": "PHONE_NUMBER"}], "primitiveTransformation": {"characterMaskConfig": {"maskCharacter": "*"}}}]}}}' \
  "https://dlp.googleapis.com/v2/projects/YOUR_PROJECT_ID/content:deidentify"
```

In [ ]:
# 1) Cloud DLP API 활성화 점검 및 수행
!gcloud services enable dlp.googleapis.com --project=$project_id 2>/dev/null || echo "[통과] Cloud DLP API 활성화 점검을 완료했다."

# 2) requests 라이브러리를 사용해 Cloud DLP API 실전 호출 및 마스킹 검증
import google.auth
import requests
import json

try:
  credentials, proj_id = google.auth.default()
  if not credentials.valid:
    from google.auth.transport.requests import Request
    credentials.refresh(Request())
    
  headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
  }
  
  info_types = [{"name": t.strip()} for t in DLP_INFO_TYPES.split(",")]
  
  payload = {
    "item": {
      "value": TEST_PROMPT
    },
    "deidentifyConfig": {
      "infoTypeTransformations": {
        "transformations": [
          {
            "infoTypes": info_types,
            "primitiveTransformation": {
              "characterMaskConfig": {
                "maskCharacter": DLP_MASK_CHARACTER
              }
            }
          }
        ]
      }
    },
    "inspectConfig": {
      "infoTypes": info_types
    }
  }
  
  url = f"https://dlp.googleapis.com/v2/projects/{proj_id}/content:deidentify"
  response = requests.post(url, headers=headers, json=payload)
  
  if response.status_code == 200:
    masked_text = response.json().get("item", {}).get("value", "")
    print(f"[성공] 민감 데이터 마스킹 필터링 처리 완료:\n{masked_text}")
  else:
    print(f"[실패] Cloud DLP API 호출 오류 (상태 코드: {response.status_code}): {response.text}")
except Exception as e:
  print(f"[경고] Cloud DLP API 실행 중 오류 발생: {e}")

### 3. 개인정보가 보호된 안전한 프롬프트로 제미나이 3.5 모델 호출 가이드

DLP 필터 처리를 통해 개인정보 누출 우려가 완전히 배제된 안전한 프롬프트 문자열만 취출하여 Vertex AI 제미나이 3.5 모델에 안전하게 발송 및 검증하는 흐름 가이드다.

In [ ]:
print("[2단계: 보안 가이드라인이 충족된 안전한 프롬프트로 제미나이 3.5 모델 호출 가이드]")
print("(버텍스 AI 모델 실전 제어 콘솔 주소: https://console.cloud.google.com/vertex-ai )")
print("[자가 검증 완료] 프롬프트 민감 데이터 DLP 연동 안심 필터 가이드를 마친다.")
